In [13]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
import ast
from googletrans import Translator
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from transformers import CLIPTokenizer, CLIPTextModel
from openai import OpenAI
import json
import re
import anthropic
import torch
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
from vertexai.preview.generative_models import GenerativeModel
import copy

In [2]:
artnet_2024 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [3]:
artnet_2024.columns

Index(['artwork id', 'artist id', 'first', 'last', 'nationality', 'year born',
       'year died', 'title', 'workyear modifier', 'workyear from',
       'workyear to', 'est lo usd', 'est hi usd', 'sale price usd', 'artist',
       'price'],
      dtype='object')

# Test on Consistency

In [4]:
Artist_name = "Käthe Kollwitz"

In [5]:
artnet_2024_Renoir = artnet_2024[artnet_2024["artist"] == Artist_name].reset_index(drop=True)

In [6]:
artnet_2024_limit=copy.deepcopy(artnet_2024_Renoir)

In [7]:
artnet_2024_limit

,artwork id,artist id,first,last,nationality,year born,year died,title,workyear modifier,workyear from,workyear to,est lo usd,est hi usd,sale price usd,artist,price
0,424044055,3278239,Käthe,Kollwitz,German,1867,1945,Städtisches Obdach,NaN,1926,NaN,4000.000000,5000.000000,3819.000000,Käthe Kollwitz,6341.864960
1,424049391,3278239,Käthe,Kollwitz,German,1867,1945,"Ein Weberaufstand (portfolio of 6, incl. 3 lit...",NaN,1897,1898.0,631.313131,631.313131,631.313131,Käthe Kollwitz,1048.364134
2,424050836,3278239,Käthe,Kollwitz,German,1867,1945,Ruf des Todes pl.8 (from Tod),NaN,1934,1935.0,2904.040404,2904.040404,3535.353535,Käthe Kollwitz,5870.839148
3,424050837,3278239,Käthe,Kollwitz,German,1867,1945,Mutter mit Jungen,NaN,1931,NaN,2020.202020,2020.202020,4166.666667,Käthe Kollwitz,6919.203282
4,424057623,3278239,Käthe,Kollwitz,German,1867,1945,Schwangere Frau,NaN,1910,NaN,1893.222264,1893.222264,1262.148176,Käthe Kollwitz,2095.934353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2748,444256792,3278239,Käthe,Kollwitz,German,1867,1945,Heimarbeit,NaN,1925,NaN,1904.157410,1904.157410,3438.061991,Käthe Kollwitz,3438.061991
2749,444256795,3278239,Käthe,Kollwitz,German,1867,1945,Frau an der Wiege,CIRCA,1897,NaN,317.359568,317.359568,264.466307,Käthe Kollwitz,264.466307
2750,444256796,3278239,Käthe,Kollwitz,German,1867,1945,Brustbild einer Arbeiterfrau mit blauem Tuch,NaN,1903,NaN,317.359568,317.359568,264.466307,Käthe Kollwitz,264.466307
2751,444256797,3278239,Käthe,Kollwitz,German,1867,1945,Die Pflüger,NaN,1907,NaN,423.146091,423.146091,396.699460,Käthe Kollwitz,396.699460


In [8]:
with open("D:\\MissTiny\\GitHub\\Creativity_Chess\\Token_Key\\claude_api.txt", "r", encoding="utf-8") as file:
    claude_api = file.read()

In [9]:
claude_client = anthropic.Anthropic(api_key=claude_api)

In [10]:
def construct_answer(response):
    blocks = response.content
    text_blocks = [b for b in blocks if getattr(b, "type", None) == "text"]
    full_text = "".join(b.text for b in text_blocks)
    art_marker = "**Artistic Value:"
    cre_marker = "**Creativity:"
    art_start = full_text.find(art_marker)
    cre_start = full_text.find(cre_marker)
    
    art_text = full_text[art_start + len(art_marker):cre_start].strip()
    cre_text  = full_text[cre_start + len(cre_marker):].strip()
    
    answer_text=f"""**Artistic Value:{art_text}
    
**Creativity:{cre_text}"""
    return answer_text

In [11]:
def annotation(i,content):
    row=[content["artwork id"],content["title"],content['first'],content['last'],content['workyear from'],content["nationality"]]

    instruction_message=fr"""You are an expert in artwork criticism, art history, and creativity evaluation.
    Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
    - Title: \'{content["title"]}\' 
    - Artist:{content['first']} {content['last']} ({content["nationality"]})
    - Year of Creation:{content['workyear from']} 
    
    Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
    Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
    - An artwork is creative only if it is new, valuable, and surprising compared to prior artworks
    
    OUTPUT FORMAT(around 150 words each)

    **Artistic Value: <High or Low>**
    <Your summary here>
    
    **Creativity: <Yes or No>**
    <Your summary here>
    """ 
    response = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",      # claude-3-5-haiku-latest
        max_tokens=1000,
        tools= [{
                "type": "web_search_20250305",
                "name": "web_search",
                "max_uses": 1
            }],
        messages=[{
                "role": "user",
                "content": instruction_message,
            }
        ]
         
    )
    
    answer = construct_answer(response)
    artistic_answer = re.search(r"\*\*Artistic Value:\s*([^*]+)\*\*", answer)
    artistic_comment = re.search(r"\*\*Artistic Value:[^*]+\*\*\s*(.*?)\n\n\*\*Creativity", answer, re.S)

    creativity_answer = re.search(r"\*\*Creativity:\s*([^*]+)\*\*", answer)
    creativity_comment = re.search(r"\*\*Creativity:[^*]+\*\*\s*(.*)", answer, re.S)

    if not (artistic_answer and artistic_comment and creativity_answer and creativity_comment):
        row=[content["artwork id"], content["title"], content['first'], content['last'], content['workyear from'], content["nationality"],"","","",""]
        return i, row, answer
    
    result = {
        "artistic_value_answer": artistic_answer.group(1).strip() if artistic_answer else "",
        "artistic_value_comment": artistic_comment.group(1).strip() if artistic_comment else "",
        "creativity_answer": creativity_answer.group(1).strip() if creativity_answer else "",
        "creativity_comment": creativity_comment.group(1).strip() if creativity_comment else ""
    }

    row.append(result['artistic_value_answer'])
    row.append(result['artistic_value_comment'])
    row.append(result['creativity_answer'])
    row.append(result['creativity_comment'])
    return i, row, ""

In [14]:
number_size= 1000
# number_size=df_raw.shape[0]
# range_start = 30000
range_start =0
range_end = min(range_start+number_size,artnet_2024_limit.shape[0])
N = min(number_size, artnet_2024_limit.shape[0]-range_start)

In [15]:
error_output=[]
annotation_result = pd.DataFrame({
    "artwork id": np.full(number_size, np.nan, dtype=int),
    "title": [""] * number_size,
    "first": [""] * number_size,
    "last": [""] * number_size,
    "workyear from":np.full(number_size, np.nan, dtype=int),
    "nationality":[""] * number_size,
    "artistic_value_answer": [""] * number_size,
    "artistic_value_comment":[""] * number_size,
    "creativity_answer": [""] * number_size,
    "creativity_comment": [""] * number_size})
max_workers = 20  # tune to your CPU cores
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Parallel computation starts")
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {
        ex.submit(annotation, i,
                   {
                "artwork id": artnet_2024_limit.iloc[i]["artwork id"],
                "title": artnet_2024_limit.iloc[i]["title"],
                "first": artnet_2024_limit.iloc[i]["first"],
                "last": artnet_2024_limit.iloc[i]["last"],
                "workyear from": artnet_2024_limit.iloc[i]["workyear from"],
                "nationality": artnet_2024_limit.iloc[i]["nationality"]}
                ): i
        for i in range(range_start,range_end)
    }
    
    completed = 0
    for fut in as_completed(futures):
        i,result,error = fut.result()
        annotation_result.loc[i-range_start]  = result
        if error !="":
            error_output.append(error)
        completed += 1
        if completed % 50 == 0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{len(artnet_2024_limit)}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: There are {len(error_output)} errors")

C:\Users\MissTiny\anaconda3\envs\Creativity\Lib\site-packages\numpy\_core\numeric.py:353: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(a, fill_value, casting='unsafe')


2025-12-16 00:12:12: Parallel computation starts
2025-12-16 00:12:40: Completed 50/2753
2025-12-16 00:13:08: Completed 100/2753
2025-12-16 00:13:35: Completed 150/2753
2025-12-16 00:14:04: Completed 200/2753
2025-12-16 00:14:32: Completed 250/2753
2025-12-16 00:15:10: Completed 300/2753
2025-12-16 00:15:39: Completed 350/2753
2025-12-16 00:16:07: Completed 400/2753
2025-12-16 00:16:34: Completed 450/2753
2025-12-16 00:17:01: Completed 500/2753
2025-12-16 00:17:26: Completed 550/2753
2025-12-16 00:17:51: Completed 600/2753
2025-12-16 00:18:14: Completed 650/2753
2025-12-16 00:18:39: Completed 700/2753
2025-12-16 00:19:02: Completed 750/2753
2025-12-16 00:19:24: Completed 800/2753
2025-12-16 00:19:48: Completed 850/2753
2025-12-16 00:20:11: Completed 900/2753
2025-12-16 00:20:32: Completed 950/2753
2025-12-16 00:20:56: Completed 1000/2753
2025-12-16 00:20:56: Ends
2025-12-16 00:20:56: There are 58 errors


In [76]:
annotation_result.to_excel(f"comment_annotation_claude_{Artist_name.split(" ")[-1]}.xlsx",index=False)

In [ ]:
annotation_result

## Check Error

In [67]:
empty_idx = annotation_result.index[annotation_result["artistic_value_answer"] == ""].tolist()

In [68]:
annotation_result.iloc[empty_idx]

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer,artistic_value_comment,creativity_answer,creativity_comment
824,425568275,Hamburger Kneipe,Käthe,Kollwitz,1901,German,,,,


In [69]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: There are {len(empty_idx)} errors")

2025-12-16 02:23:26: There are 1 errors


In [70]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start")
count= 0
for error_index in empty_idx:
    if count%10 ==0:
        print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Current at {count}")
    content = {
                "artwork id": artnet_2024_limit.iloc[error_index]["artwork id"],
                "title": artnet_2024_limit.iloc[error_index]["title"],
                "first": artnet_2024_limit.iloc[error_index]["first"],
                "last": artnet_2024_limit.iloc[error_index]["last"],
                "workyear from": artnet_2024_limit.iloc[error_index]["workyear from"],
                "nationality": artnet_2024_limit.iloc[error_index]["nationality"]}
    _,result,error = annotation(error_index,content)
    annotation_result.loc[error_index,:] =result
    count+=1
    # if count ==10:
    #     break
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")

2025-12-16 02:23:26: Start
2025-12-16 02:23:26: Current at 0
2025-12-16 02:23:40: Ends


In [71]:
empty_idx

[824]

In [74]:
annotation_result.iloc[empty_idx]

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer,artistic_value_comment,creativity_answer,creativity_comment
824,425568275,Hamburger Kneipe,Käthe,Kollwitz,1901,German,,,,


In [75]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: There are {len(annotation_result.index[annotation_result["artistic_value_answer"] == ""].tolist())} errors")

2025-12-16 02:30:36: There are 1 errors


In [66]:
annotation_result.shape

(1000, 10)

# Error Fix

In [140]:
artnet_2024_Picasso = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_Picasso.xlsx")

In [141]:
annotation_result = pd.read_excel("comment_annotation_claude_final.xlsx")

In [143]:
sum(annotation_result['artwork id'].duplicated())

535

In [152]:
sum(artnet_2024_Picasso.iloc[0:5000]['artwork id'].duplicated())

0

In [145]:
check_point = annotation_result['artwork id'] == artnet_2024_Picasso.iloc[0:5000]['artwork id']

In [234]:
a = artnet_2024_Picasso.iloc[0:5000][annotation_result['artwork id'] != artnet_2024_Picasso.iloc[0:5000]['artwork id']]

In [235]:
a.head()

,artwork id,artist id,first,last,nationality,year born,year died,title,workyear modifier,workyear from,workyear to,est lo usd,est hi usd,sale price usd,artist,price


In [159]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start")
count=0
for i in range(5000):
    if not check_point[i]:
        content = {
            "artwork id": artnet_2024_Picasso.iloc[i]["artwork id"],
            "title": artnet_2024_Picasso.iloc[i]["title"],
            "first": artnet_2024_Picasso.iloc[i]["first"],
            "last": artnet_2024_Picasso.iloc[i]["last"],
            "workyear from": artnet_2024_Picasso.iloc[i]["workyear from"],
            "nationality": artnet_2024_Picasso.iloc[i]["nationality"]}
        _,result,error = annotation(i,content)
        annotation_result.loc[i,:] =result
        count+=1
        if count%50 ==0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Current at {count}")

print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")

2025-12-04 06:13:52: Start
2025-12-04 06:21:26: Current at 50
2025-12-04 06:28:54: Current at 100
2025-12-04 06:36:18: Current at 150
2025-12-04 06:43:44: Current at 200
2025-12-04 06:51:15: Current at 250
2025-12-04 06:58:48: Current at 300
2025-12-04 07:06:12: Current at 350
2025-12-04 07:14:00: Current at 400
2025-12-04 07:21:39: Current at 450
2025-12-04 07:29:08: Current at 500
2025-12-04 07:34:16: Ends


In [236]:
annotation_result.to_excel("comment_annotation_claude_fix.xlsx",index=False)

# Test on Prompt Sensitivity Code

In [9]:
with open("D:\\MissTiny\\GitHub\\Creativity_Chess\\Token_Key\\claude_api.txt", "r", encoding="utf-8") as file:
    claude_api = file.read()

In [12]:
content = artnet_2024_Picasso.iloc[1]

In [13]:
instruction_message=fr"""You are an expert in artwork criticism, art history, and creativity evaluation.
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: \'{content["title"]}\' 
- Artist:{content['first']} {content['last']} ({content["nationality"]})
- Year of Creation:{content['workyear from']} 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)

**Artistic Value: <High or Low>**
<Your summary here>

**Creativity: <Yes or No>**
<Your summary here>
""" 

In [14]:
print(instruction_message)

You are an expert in artwork criticism, art history, and creativity evaluation.
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: \'Mousquetaire buste\' 
- Artist:Pablo Picasso (Spanish)
- Year of Creation:1968 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)

**Artistic Value: <High or Low>**
<Your summary here>

**Creativity: <Yes or No>**
<Your summary here>



In [15]:
claude_client = anthropic.Anthropic(
    api_key=claude_api
)

In [16]:
response = claude_client.messages.create(
    model="claude-haiku-4-5-20251001",      # claude-3-5-haiku-latest
    max_tokens=1000,
    tools= [{
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 2
        }],
    messages=[{
            "role": "user",
            "content": instruction_message,
        }
    ]
     
)

In [22]:
response.content

[TextBlock(citations=None, text='I\'ll search for authoritative commentary on Picasso\'s "Mousquetaire buste" from 1968.', type='text'),
 ServerToolUseBlock(id='srvtoolu_01MKS13ZNa2Hjv6UJe9Pw4pK', input={'query': 'Picasso Mousquetaire buste 1968 artistic significance'}, name='web_search', type='server_tool_use'),
 ServerToolUseBlock(id='srvtoolu_01M2YphiN56yVEW8myJdc3At', input={'query': '"Mousquetaire buste" Picasso 1968 sculpture'}, name='web_search', type='server_tool_use'),
 ServerToolUseBlock(id='srvtoolu_01Nz4a73vguVHU5FjRK9FXT7', input={'query': 'Picasso 1968 sculpture artistic innovation late work'}, name='web_search', type='server_tool_use'),
 WebSearchToolResultBlock(content=[WebSearchResultBlock(encrypted_content='EqYcCioIChgCIiRlOWRlZGIzNi0yNDFmLTQ3NTUtYTA1Mi02OTg1OTRiNGI5MTgSDDcAfKKIHsPH6fWk+hoMw8Ej2KwRyIIRML2LIjChboDhEZ9O5t1x9tXqiKC9wSn0gigW55X8qg+nitgYZAtSTVjIU6tvbGQW3VNNRzMqqRv6vHUgzaqcwVLoIo3WouPpIDkM23+yetK5NeGSJAAdOuIJehzfzGeRi59wZK7Lnhz8TK3IzrM++bcH1CafIzd7Y2hjccLNB

In [26]:
blocks = response.content
text_blocks = [b for b in blocks if getattr(b, "type", None) == "text"]
full_text = "".join(b.text for b in text_blocks)
art_marker = "**Artistic Value:"
cre_marker = "**Creativity:"
art_start = full_text.find(art_marker)
cre_start = full_text.find(cre_marker)

art_text = full_text[art_start + len(art_marker):cre_start].strip()
cre_text  = full_text[cre_start + len(cre_marker):].strip()

answer_text=f"""
**Artistic Value:{art_text}

**Creativity:{cre_text}"""

In [27]:
print(answer_text)


**Artistic Value:High**

The Mousquetaire. Buste of 1968 belongs to a major series of paintings that Picasso executed on the theme of the musketeer, which became one of the key subjects of his late œuvre. Picasso's masterful use of simple lines and curves, basic pictorial elements, almost have a childlike directness to depict the nose and mouth, creating a multi-viewpoint perspective and frames the face of the musketeer without having to literally outline it for the viewer, rendered in cerulean blue, mauve, emerald green, and crisp ivory, boldly applied with intense confidence. These portraits of various archetypes that populated Picasso's personal mythology were part of a late flowering, a final synthesis which merged the artist's personal history with the cultural heritage of the Western artistic tradition, and developed a direct and spontaneous style that celebrated the act of artistic creation. The surface of the late paintings has a freedom, a plasticity, that was never there bef

In [53]:
blocks = response.content
text_blocks = [b for b in blocks if getattr(b, "type", None) == "text"]
full_text = "".join(b.text for b in text_blocks)
art_marker = "Artistic Value:"
cre_marker = "Creativity:"
art_start = full_text.find(art_marker)
cre_start = full_text.find(cre_marker)

art_text = full_text[art_start + len(art_marker):cre_start].strip()
cre_text  = full_text[cre_start + len(cre_marker):].strip()

answer_text=f"""
Artistic Value:
{art_text}
Creativity:
{cre_text}"""

In [55]:
print(answer_text)


Artistic Value:
**

Gentile da Fabriano figures among the artists of the major Gothic international style, and was the foremost painter of central Italy at the beginning of the 15th century, whose surviving works are among the finest examples of the International Gothic style. While the specific 1420 painting lacks documented scholarly analysis, Gentile's Madonna compositions from this period demonstrate significant technical achievement. The Virgin becomes a figure more maternally oriented and transforms into the Virgin of Humility, seated on the ground. His work marks a turning point in the artist's career: a mastery of figural construction and an effect of monumentality achieved through simplicity in the silhouette of the Madonna and Child and a richness in the rhythmic articulation of drapery. This demonstrated his improved naturalistic technique with the use of light to create dimensions and perspective, with contrasting light bringing figures to life and making them appear more 

In [117]:
possible_prompts=[]

In [56]:
prompt1=f"""You are an expert in artwork criticism, art history, and creativity evaluation.
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: \'{row.title}\' 
- Artist:{row['first']} {row['last']} ({row.nationality})
- Year of Creation:{row['workyear from']} 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)
Artistic Value: 
<Your summary here>
Creativity: <Yes or No>
<Your summary here>
"""

In [83]:
print(prompt1)

You are an expert in artwork criticism, art history, and creativity evaluation.
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: 'La Vierge d'humilité, tableau de dévotion' 
- Artist:Francesco di Gentile da Fabriano (Italian)
- Year of Creation:1420 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)
Artistic Value: 
<Your summary here>
Creativity: <Yes or No>
<Your summary here>



In [57]:
response1 = claude_client.messages.create(
    model="claude-haiku-4-5-20251001",      # claude-3-5-haiku-latest
    max_tokens=1000,
    tools= [{
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 2
        }],
    messages=[{
            "role": "user",
            "content": prompt1,
        }
    ]
     
)

blocks = response1.content
text_blocks = [b for b in blocks if getattr(b, "type", None) == "text"]
full_text = "".join(b.text for b in text_blocks)
art_marker = "Artistic Value:"
cre_marker = "Creativity:"
art_start = full_text.find(art_marker)
cre_start = full_text.find(cre_marker)

art_text = full_text[art_start + len(art_marker):cre_start].strip()
cre_text  = full_text[cre_start + len(cre_marker):].strip()

answer_text1=f"""
Artistic Value:
{art_text}
Creativity:
{cre_text}"""
print(answer_text1 )


Artistic Value:
**

Gentile da Fabriano's Madonna of Humility is a tempera-on-panel painting dating from around 1420–1423. Its modest scale suggests it was intended for private devotion. The picture marks a turning point in the artist's career: a mastery of figural construction and an effect of monumentality achieved through simplicity in the silhouette of the Madonna and Child and a richness in the rhythmic articulation of the drapery. The Madonna of Humility, with the Virgin sitting on a cushion over the ground, was a common theme in early 15th century western painting. Astonishing at this date is the delicate naturalism of the child, the attentive description of the plants, and the rhythmic folds of the drapery, which confer an effect of incipient movement.

**
Creativity:
No**

The Madonna of Humility iconographic type originated with Simone Martini during his Avignon period, and from 1348 it progressively replaced the Virgin in Majesty. While Gentile da Fabriano executed the moti

In [71]:
prompt2=f"""Act as an expert in artwork criticism, art history, and creativity evaluation.
Using web search, summarize in English the artistic value and creative significance of {row['first']} {row['last']}’s {row['workyear from']}  artwork “{row.title}”.
Note that an artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT: (~150 words each)
Artistic Value:
<text>

Creativity: <Yes or No>
<text>
"""

In [72]:
response2 = claude_client.messages.create(
    model="claude-haiku-4-5-20251001",      # claude-3-5-haiku-latest
    max_tokens=1000,
    tools= [{
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 2
        }],
    messages=[{
            "role": "user",
            "content": prompt2,
        }
    ]
     
)

blocks = response2.content
text_blocks = [b for b in blocks if getattr(b, "type", None) == "text"]
full_text = "".join(b.text for b in text_blocks)
art_marker = "Artistic Value:"
cre_marker = "Creativity:"
art_start = full_text.find(art_marker)
cre_start = full_text.find(cre_marker)

art_text = full_text[art_start + len(art_marker):cre_start].strip()
cre_text  = full_text[cre_start + len(cre_marker):].strip()

answer_text2=f"""
Artistic Value:
{art_text}
Creativity:
{cre_text}"""
print(answer_text2 )


Artistic Value:
**

The work exemplifies International Gothic refinement characteristic of early 15th-century devotional painting. These private devotional tableaux counterbalanced the solemnity of Majesty representations, illustrating the Virgin's chastity symbolized by the enclosed garden. The painting likely employs tempera and gold leaf on panel, typical of Fabriano's workshop practice. The Virgin becomes a more maternal figure transformed gradually into the Virgin of Humility, seated on the ground, representing a significant shift from medieval hieratic conventions toward humanized spirituality. The refined decorative linearity and precious materials demonstrate technical mastery, though the workshop attribution suggests variable execution quality within the composition.

**
Creativity:
No**

While the Virgin of Humility itself was innovative iconographically, this iconography was invented in Siena around 1340 by Simone Martini during his stay in Avignon. By 1420, this subject ty

In [73]:
prompt3=f"""ou are a general expert in artwork criticism, art history, and creativity evaluation.
Using web search, analyze the artwork:

- Title: '{row.title}'
- Artist: {row['first']} {row['last']} ({row.nationality})
- Year: {row['workyear from']}

Based ONLY on web search results, provide two ~150-word sections evaluating:
1. Artistic Value
2. Creativity

Note that an artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT:
Artistic Value:
<text>
Creativity: <Yes or No>
<text>
"""

In [74]:
response3 = claude_client.messages.create(
    model="claude-haiku-4-5-20251001",      # claude-3-5-haiku-latest
    max_tokens=1000,
    tools= [{
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 2
        }],
    messages=[{
            "role": "user",
            "content": prompt3,
        }
    ]
     
)

blocks = response3.content
text_blocks = [b for b in blocks if getattr(b, "type", None) == "text"]
full_text = "".join(b.text for b in text_blocks)
art_marker = "Artistic Value:"
cre_marker = "Creativity:"
art_start = full_text.find(art_marker)
cre_start = full_text.find(cre_marker)

art_text = full_text[art_start + len(art_marker):cre_start].strip()
cre_text  = full_text[cre_start + len(cre_marker):].strip()

answer_text3=f"""
Artistic Value:
{art_text}
Creativity:
{cre_text}"""
print(answer_text3 )


Artistic Value:
**
Gentile da Fabriano figures among the major artists of the International Gothic style, and this painting exemplifies the refined devotional tradition of his era. The Virgin of Humility represents a transformation from the rigid medieval Majestas toward a more maternal figure depicted seated at ground level, reflecting humanistic values emerging in early 15th-century Italian art. Gentile's improved naturalistic technique employed light to create dimensions and perspective, with contrasting light bringing figures to life and making them appear more naturally human. The painting's intimate devotional scale and careful execution demonstrate masterful technical skill typical of Gentile's Florence period around 1420. Though modest in scale compared to his grand altarpieces, this work exemplifies the refinement of International Gothic painting at its zenith, characterized by elegant composition and spiritual contemplation.

**
Creativity:
No**

While artistically accomplis

In [75]:
prompt4=f"""Assume the role of a comprehensive art expert with deep knowledge of global art history, visual analysis, and creativity theory.
Using web search, gather authoritative scholarly commentary on:

- '{row.title}' by {row['first']} {row['last']} ({row.nationality}), created in {row['workyear from']}.

Based strictly on verified search results, summarize the artwork’s:
1. Artistic Value — aesthetic qualities, emotional resonance, technical execution, symbolism, and critical reception.
2. Creativity — an artwork is creative only if it is new, valuable, and surprising compared to prior artworks.

Write ~150 words per section, in English.

FORMAT:
Artistic Value:
<text>

Creativity: <Yes or No>
<text>
"""

In [76]:
response4 = claude_client.messages.create(
    model="claude-haiku-4-5-20251001",      # claude-3-5-haiku-latest
    max_tokens=1000,
    tools= [{
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 2
        }],
    messages=[{
            "role": "user",
            "content": prompt4,
        }
    ]
     
)

blocks = response4.content
text_blocks = [b for b in blocks if getattr(b, "type", None) == "text"]
full_text = "".join(b.text for b in text_blocks)
art_marker = "Artistic Value:"
cre_marker = "Creativity:"
art_start = full_text.find(art_marker)
cre_start = full_text.find(cre_marker)

art_text = full_text[art_start + len(art_marker):cre_start].strip()
cre_text  = full_text[cre_start + len(cre_marker):].strip()

answer_text4=f"""
Artistic Value:
{art_text}
Creativity:
{cre_text}"""
print(answer_text4)


Artistic Value:
r authoritative scholarship on this specific work by Francesco di Gentile da Fabriano.I appreciate your request, but I must inform you of a limitation with the search results. While I found information about Francesco di Gentile da Fabriano and his broader artistic practice around 1420, the search results do not contain substantive scholarly commentary specifically about "La Vierge d'humilité, tableau de dévotion" (1420).

The search results confirm that the work exists and was catalogued in an auction on December 15, 2010, and that Gentile incorporated pseudo-Arabic into his Madonna of the Humility (c. 1420). However, the available results do not provide detailed authoritative analysis of:

- This specific painting's aesthetic qualities, technical execution, or symbolism
- Its critical reception or scholarly interpretation
- How it compares to other Madonna of Humility compositions of the period
- Whether it represents a creative innovation

**What I can confirm:** Du

In [77]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

In [78]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = text_model.to(device)

In [79]:
texts=[answer_text1,answer_text2,answer_text3,answer_text4]

In [80]:
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
text_embeds = outputs.pooler_output
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
text_embeds=text_embeds.cpu().numpy()

In [81]:
pairwise = cosine_similarity(text_embeds)

In [82]:
pairwise

array([[0.99999994, 0.7179121 , 0.8479196 , 0.62829363],
       [0.7179121 , 1.        , 0.7724914 , 0.63452005],
       [0.8479196 , 0.7724914 , 1.0000002 , 0.6945832 ],
       [0.62829363, 0.63452005, 0.6945832 , 0.99999976]], dtype=float32)